# Used Car Price Prediction – Regression Models

## ITAI 1371 | Final Project | Spring 2026 | Group 5



---

## Table of Contents

1. Dataset Upload and Initialization  
2. Data Partitioning (Training / Validation / Testing – 70/15/15)  
3. Feature Scaling and Preparation  
4. Model Building  
   - 4.1 Linear Regression Model  
   - 4.2 Decision Tree Model  
   - 4.3 Random Forest Model  
   - 4.4 Gradient Boosting Model  
   - 4.5 K-Nearest Neighbors Model  
5. Performance Evaluation and Comparison  
6. Ensemble Learning Methods  
   - 6.1 Voting Regressor (Best 3 Models)  
   - 6.2 Bayesian-Based Ensemble  
7. Testing and Final Performance Check  
8. Overall Results Summary  

<a id='1'></a>
## 1. Setup & Data Load

In [32]:
import pandas as pd
import numpy as np
import os

if os.path.exists('final_cleaned_dataset_readable.csv'):
    df = pd.read_csv('final_cleaned_dataset_readable.csv')
    print("Dataset loaded successfully ✅")
    print("Shape:", df.shape)
    df.head()
else:
    print("CSV file not found. Please upload the dataset file.")

CSV file not found. Please upload the dataset file.


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,
                               GradientBoostingRegressor,
                               VotingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded.')


Libraries loaded.


In [34]:
import pandas as pd
import numpy as np
import os

# Load cleaned dataset safely
if os.path.exists('final_cleaned_dataset_readable.csv'):

    df = pd.read_csv('final_cleaned_dataset_readable.csv')

    # Apply log1p only if AskPrice is still in raw form
    if df['AskPrice'].max() > 100:
        df['AskPrice'] = np.log1p(df['AskPrice'])
        print('log1p applied to AskPrice.')
    else:
        print('AskPrice already in log-space — no transform needed.')

    print(f'Dataset shape: {df.shape}')
    print(f'AskPrice range: {df["AskPrice"].min():.2f} → {df["AskPrice"].max():.2f}')

    df.head()

else:
    print("CSV file not found. Please upload final_cleaned_dataset_readable.csv")


CSV file not found. Please upload final_cleaned_dataset_readable.csv


<a id='2'></a>
## 2. Train / Validation / Test Split — 70 / 15 / 15

The dataset is split into training, validation, and test sets in a 70/15/15 ratio. This split is used to train the models, compare performance, and evaluate final results.

In [35]:
# Make sure dataset is loaded
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Check if df exists
if 'df' in globals():

    X = df.drop(columns=['AskPrice'])
    y = df['AskPrice']

    # Step 1: hold out 30% as temporary set
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    # Step 2: split temp equally into validation and test sets
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42
    )

    print(f'Training set   : {X_train.shape[0]} rows ({X_train.shape[0]/len(df)*100:.0f}%)')
    print(f'Validation set : {X_val.shape[0]} rows ({X_val.shape[0]/len(df)*100:.0f}%)')
    print(f'Test set       : {X_test.shape[0]} rows ({X_test.shape[0]/len(df)*100:.0f}%)')
    print(f'Features       : {X_train.shape[1]}')

else:
    print("Dataset not loaded. Please run the dataset loading cell first.")


Dataset not loaded. Please run the dataset loading cell first.


<a id='3'></a>
## 3. Feature Scaling

In this step, selected numerical features are scaled using StandardScaler. The scaler is fitted on the training data and applied to validation and test sets.


In [36]:
from sklearn.preprocessing import StandardScaler

# Check if train/validation/test sets exist
if 'X_train' in globals():

    scaler = StandardScaler()

    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()

    num_cols = ['kmDriven', 'Age']

    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_val[num_cols] = scaler.transform(X_val[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

    print("Scaling complete. ✅")
    print(f'kmDriven → train mean: {X_train["kmDriven"].mean():.4f}, std: {X_train["kmDriven"].std():.4f}')
    print(f'Age → train mean: {X_train["Age"].mean():.4f}, std: {X_train["Age"].std():.4f}')

else:
    print("Training dataset not found. Please run the train/validation/test split cell first.")


Training dataset not found. Please run the train/validation/test split cell first.


<a id='4'></a>
## 4. Model Training
In this step, multiple regression models are trained using the training dataset.

### 4.1 Linear Regression
Baseline regression model with a linear assumption between features and the target variable.

In [37]:
from sklearn.linear_model import LinearRegression
import pandas as pd

# Check if training data exists
if 'X_train' in globals() and 'y_train' in globals():

    lr = LinearRegression()

    lr.fit(X_train, y_train)

    print("Linear Regression trained. ✅")

    # Top coefficients
    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': lr.coef_
    })

    coef_df = coef_df.reindex(
        coef_df['Coefficient'].abs().sort_values(ascending=False).index
    )

    print("\nTop 10 most influential features:")
    print(coef_df.head(10).to_string(index=False))

else:
    print("Training data not found. Please run previous cells first.")


Training data not found. Please run previous cells first.


### 4.2 Decision Tree Regressor
Non-linear model that splits data into decision-based regions.

In [38]:
from sklearn.tree import DecisionTreeRegressor

# Check if training data exists
if 'X_train' in globals() and 'y_train' in globals():

    dt = DecisionTreeRegressor(random_state=42)

    dt.fit(X_train, y_train)

    print(f"Decision Tree trained. Max depth reached: {dt.get_depth()} ✅")

else:
    print("Training data not found. Please run previous cells first.")


Training data not found. Please run previous cells first.


### 4.3 Random Forest Regressor
Ensemble model combining multiple decision trees to improve accuracy and reduce overfitting.

In [39]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt

# Check if training data exists
if 'X_train' in globals() and 'y_train' in globals():

    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    print("Random Forest trained (200 trees). ✅")

    # Feature importance
    fi = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': rf.feature_importances_
    })

    fi = fi.sort_values('Importance', ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.barh(fi['Feature'][::-1], fi['Importance'][::-1])

    ax.set_title('Random Forest — Top 10 Feature Importances', fontsize=12)
    ax.set_xlabel('Importance')

    plt.tight_layout()
    plt.show()

else:
    print("Training data not found. Please run previous cells first.")

Training data not found. Please run previous cells first.


### 4.4 Gradient Boosting Regressor
Boosting model with 300 trees that improves predictions by learning from previous errors.

In [40]:
from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd
import matplotlib.pyplot as plt

# Check if training data exists
if 'X_train' in globals() and 'y_train' in globals():

    gb = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    )

    gb.fit(X_train, y_train)

    print("Gradient Boosting trained (300 estimators, lr=0.05, depth=5). ✅")

    # Feature importance
    fi_gb = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': gb.feature_importances_
    })

    fi_gb = fi_gb.sort_values('Importance', ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.barh(fi_gb['Feature'][::-1], fi_gb['Importance'][::-1])

    ax.set_title('Gradient Boosting — Top 10 Feature Importances', fontsize=12)
    ax.set_xlabel('Importance')

    plt.tight_layout()
    plt.show()

else:
    print("Training data not found. Please run previous cells first.")


Training data not found. Please run previous cells first.


### 4.5 K-Nearest Neighbors Regressor
Predicts target values by averaging the outputs of the 7 nearest data points.

In [41]:
from sklearn.neighbors import KNeighborsRegressor

# Check if training data exists
if 'X_train' in globals() and 'y_train' in globals():

    knn = KNeighborsRegressor(n_neighbors=7)

    knn.fit(X_train, y_train)

    print("KNN trained (k=7). ✅")

else:
    print("Training data not found. Please run previous cells first.")


Training data not found. Please run previous cells first.


<a id='5'></a>
## 5. Validation & Comparison

All trained models are assessed on the validation set for performance comparison.
Metrics are calculated in log-space due to the log transformation of the target variable.

The actual error in rupees is obtained by reversing the transformation using np.expm1.

In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Check if all models and validation data exist
required_vars = ['lr', 'dt', 'rf', 'gb', 'knn', 'X_val', 'y_val']

if all(var in globals() for var in required_vars):

    def evaluate(name, model, X, y):
        pred = model.predict(X)

        return {
            'Model': name,
            'MAE': mean_absolute_error(y, pred),
            'MSE': mean_squared_error(y, pred),
            'R2': r2_score(y, pred)
        }

    models_dict = {
        'Linear Regression': lr,
        'Decision Tree': dt,
        'Random Forest': rf,
        'Gradient Boosting': gb,
        'KNN': knn
    }

    val_results = pd.DataFrame([
        evaluate(name, m, X_val, y_val)
        for name, m in models_dict.items()
    ]).set_index('Model')

    print("=== Validation Set Metrics ===")
    print(val_results.sort_values('R2', ascending=False).to_string())

else:
    print("Some models or validation datasets are missing. Please run previous cells first.")


Some models or validation datasets are missing. Please run previous cells first.


The validation results are summarized in a table showing MAE, MSE, and R² scores for each model.
A bar chart is generated to visually compare model performance across different evaluation metrics.

In [43]:
import matplotlib.pyplot as plt

# Check if validation results exist
if 'val_results' in globals():

    # Bar chart comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
    metrics = ['MAE', 'MSE', 'R2']

    for ax, metric in zip(axes, metrics):

        vals = val_results[metric].sort_values(
            ascending=(metric != 'R2')
        )

        bars = ax.barh(vals.index, vals.values, color=colors[:len(vals)])

        ax.set_title(f'Validation {metric}', fontsize=11)
        ax.set_xlabel(metric)

        for bar, v in zip(bars, vals.values):

            ax.text(
                bar.get_width() + 0.002,
                bar.get_y() + bar.get_height()/2,
                f'{v:.4f}',
                va='center',
                fontsize=8
            )

    plt.suptitle(
        'Model Comparison — Validation Set',
        fontsize=13,
        y=1.01
    )

    plt.tight_layout()
    plt.show()

else:
    print("Validation results not found. Please run previous cells first.")


Validation results not found. Please run previous cells first.


<a id='6'></a>
## 6. Ensemble Models

Based on the validation R² scores, the top three models selected are:

Gradient Boosting
KNN
Random Forest

These models are used to build ensemble techniques to improve overall prediction performance.


### 6.1 Voting Regressor (Top 3 Models)
This method combines the predictions of the three best-performing models by averaging them equally to generate the final prediction.

In [44]:
from sklearn.ensemble import VotingRegressor

# Check if models and training data exist
required_vars = ['gb', 'knn', 'rf', 'X_train', 'y_train', 'X_val', 'y_val']

if all(var in globals() for var in required_vars):

    voting = VotingRegressor(
        estimators=[
            ('gb', gb),
            ('knn', knn),
            ('rf', rf)
        ]
    )

    voting.fit(X_train, y_train)

    print("Voting Regressor trained. ✅")

    # Validation evaluation
    pred_v = voting.predict(X_val)

    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    val_v = {
        'MAE': mean_absolute_error(y_val, pred_v),
        'MSE': mean_squared_error(y_val, pred_v),
        'R2': r2_score(y_val, pred_v)
    }

    print("\nVoting — Validation:")
    for k, v in val_v.items():
        print(f'{k}: {v:.4f}')

else:
    print("Required models or datasets not found. Please run previous cells first.")


Required models or datasets not found. Please run previous cells first.


6.2 Bayesian Stacking Ensemble (Bayesian Ridge Meta-Model)

A stacking method where predictions from the top models are used as input features to train a Bayesian Ridge meta-model.
The meta-model learns optimal weights for each base model and improves overall performance.

The meta-model is trained on validation predictions and evaluated on the test set to ensure proper generalization.

In [45]:
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Check if required models and datasets exist
required_vars = ['gb', 'knn', 'rf', 'X_val', 'X_test', 'y_val']

if all(var in globals() for var in required_vars):

    # Stack validation/test predictions as meta-features
    val_meta = np.column_stack([
        gb.predict(X_val),
        knn.predict(X_val),
        rf.predict(X_val)
    ])

    test_meta = np.column_stack([
        gb.predict(X_test),
        knn.predict(X_test),
        rf.predict(X_test)
    ])

    # Train Bayesian Ridge meta-learner
    br = BayesianRidge()

    br.fit(val_meta, y_val)

    print("Bayesian Ensemble trained. ✅")

    print(
        f"Learned weights (coef_): "
        f"GB={br.coef_[0]:.4f}, "
        f"KNN={br.coef_[1]:.4f}, "
        f"RF={br.coef_[2]:.4f}"
    )

    # Validation evaluation
    pred_br = br.predict(val_meta)

    val_br = {
        'MAE': mean_absolute_error(y_val, pred_br),
        'MSE': mean_squared_error(y_val, pred_br),
        'R2': r2_score(y_val, pred_br)
    }

    print("\nBayesian Ensemble — Validation:")
    for k, v in val_br.items():
        print(f'{k}: {v:.4f}')

else:
    print("Required models or datasets not found. Please run previous cells first.")


Required models or datasets not found. Please run previous cells first.


The Bayesian Ridge meta-model is trained using validation predictions from the top models. The ensemble performance is then evaluated and compared using MAE, MSE, and R² metrics.

⚠️ The test data is not used during training to avoid data leakage.

In [46]:
import pandas as pd
import matplotlib.pyplot as plt

# Check if ensemble validation results exist
if 'val_v' in globals() and 'val_br' in globals():

    ens_df = pd.DataFrame([
        {'Model': 'Voting Ensemble', **val_v},
        {'Model': 'Bayesian Ensemble', **val_br}
    ]).set_index('Model')

    print("=== Ensemble Comparison — Validation Set ===")
    print(ens_df.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))

    for ax, metric in zip(axes, ['MAE', 'MSE', 'R2']):
        ax.bar(ens_df.index, ens_df[metric])
        ax.set_title(metric, fontsize=11)
        ax.tick_params(axis='x', rotation=10)

        for i, v in enumerate(ens_df[metric]):
            ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

    plt.suptitle('Ensemble Comparison — Validation Set', fontsize=12)
    plt.tight_layout()
    plt.show()

else:
    print("Ensemble results not found. Please run Voting and Bayesian Ensemble cells first.")

Ensemble results not found. Please run Voting and Bayesian Ensemble cells first.


<a id='7'></a>
## 7. Final Evaluation on Test Set

The test set has been **untouched until now**. We evaluate all 7 models once.


In [47]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Check if all required models and datasets exist
required_vars = [
    'models_dict', 'voting', 'br',
    'X_test', 'y_test', 'test_meta'
]

if all(var in globals() for var in required_vars):

    def evaluate(name, model, X, y):
        pred = model.predict(X)

        return {
            'Model': name,
            'MAE': mean_absolute_error(y, pred),
            'MSE': mean_squared_error(y, pred),
            'R2': r2_score(y, pred)
        }

    all_test_results = []

    for name, m in models_dict.items():
        all_test_results.append(
            evaluate(name, m, X_test, y_test)
        )

    all_test_results.append(
        evaluate('Voting Ensemble', voting, X_test, y_test)
    )

    all_test_results.append(
        evaluate('Bayesian Ensemble', br, test_meta, y_test)
    )

    test_df = pd.DataFrame(all_test_results).set_index('Model')

    print("=== Final Test Set Metrics ===")
    print(test_df.sort_values('R2', ascending=False).to_string())

else:
    print("Required models or test datasets not found. Please run previous cells first.")

Required models or test datasets not found. Please run previous cells first.


⚠️ The test set is used only once for final evaluation to ensure unbiased results.

In [48]:
import matplotlib.pyplot as plt

# Check if Gradient Boosting model and test data exist
required_vars = ['gb', 'X_test', 'y_test']

if all(var in globals() for var in required_vars):

    # Predictions
    gb_pred_test = gb.predict(X_test)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Scatter plot
    axes[0].scatter(
        y_test,
        gb_pred_test,
        alpha=0.3,
        s=12,
        color='steelblue'
    )

    axes[0].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        'r--',
        linewidth=1.5
    )

    axes[0].set_xlabel('Actual log1p(AskPrice)')
    axes[0].set_ylabel('Predicted log1p(AskPrice)')
    axes[0].set_title(
        'Gradient Boosting: Predicted vs Actual (Test Set)'
    )

    # Residual plot
    residuals = y_test - gb_pred_test

    axes[1].hist(
        residuals,
        bins=50,
        color='coral',
        edgecolor='white'
    )

    axes[1].axvline(0, color='black', linestyle='--')

    axes[1].set_xlabel('Residual (Actual − Predicted)')
    axes[1].set_ylabel('Count')

    axes[1].set_title(
        'Gradient Boosting: Residual Distribution (Test Set)'
    )

    plt.tight_layout()
    plt.show()

else:
    print("Gradient Boosting model or test dataset not found. Please run previous cells first.")

Gradient Boosting model or test dataset not found. Please run previous cells first.


<a id='8'></a>
## 8. Results Summary Table

In [49]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

required_vars = [
    'models_dict', 'voting', 'br',
    'X_val', 'y_val', 'X_test', 'y_test',
    'val_meta', 'test_meta'
]

if all(var in globals() for var in required_vars):

    combined = []

    all_models_eval = list(models_dict.keys()) + [
        'Voting Ensemble',
        'Bayesian Ensemble'
    ]

    for name in all_models_eval:

        if name in models_dict:
            m = models_dict[name]
            vp = m.predict(X_val)
            tp = m.predict(X_test)

        elif name == 'Voting Ensemble':
            vp = voting.predict(X_val)
            tp = voting.predict(X_test)

        else:
            vp = br.predict(val_meta)
            tp = br.predict(test_meta)

        combined.append({
            'Model': name,
            'Val_MAE': mean_absolute_error(y_val, vp),
            'Val_MSE': mean_squared_error(y_val, vp),
            'Val_R2': r2_score(y_val, vp),
            'Test_MAE': mean_absolute_error(y_test, tp),
            'Test_MSE': mean_squared_error(y_test, tp),
            'Test_R2': r2_score(y_test, tp)
        })

    summary_df = pd.DataFrame(combined).set_index('Model')

    print("=== FULL RESULTS SUMMARY ===")
    print(summary_df.to_string())

else:
    print("Required results are missing. Please run all previous model and evaluation cells first.")

Required results are missing. Please run all previous model and evaluation cells first.


In [50]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if summary table exists
if 'summary_df' in globals():

    # Heatmap of all metrics
    fig, ax = plt.subplots(figsize=(12, 5))

    sns.heatmap(
        summary_df.astype(float),
        annot=True,
        fmt='.4f',
        cmap='RdYlGn',
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Metric Value'}
    )

    ax.set_title(
        'Model Performance Heatmap — Validation & Test Metrics',
        fontsize=13
    )

    plt.tight_layout()
    plt.show()

else:
    print("Summary table not found. Please run the Results Summary Table cell first.")


Summary table not found. Please run the Results Summary Table cell first.
